# Echo of the Inkwell — Phase 12 Kaito Master Design (Colab)

Optional **GPU** path to generate **4 REAL master design candidates (A–D)** for Kaito.

## Hard rules
- **GPU required.** If no CUDA GPU is present, this notebook prints `COLAB_GPU_BLOCKED` and **stops**. It does **not** fall back to Pillow mock / fake images.
- Outputs are **candidates only** — never auto-selected as the production Kaito design.
- Prompts emphasize **coloring-book black-and-white line art** (closed outlines, white fill regions, no screentones).
- Metadata must record `source_type: REAL`, `production_eligible: true`, seeds, model id, and prompts.
- Owner must confirm model license before any KDP production use (`reports/model-licensing.json`).

## Import back to local project

```bash
python scripts/import_colab_kaito_designs.py path/to/kaito_designs_*.zip
```

Optional: add `--mark-production-eligible` after you have reviewed the art.
Then open Streamlit → **Kaito Master Design** and select a winner manually.


## 1. Install dependencies

Install torch (CUDA), diffusers, and helpers. Use a **GPU runtime** (Runtime → Change runtime type → GPU).

**Recommended open anime/manga-capable model (user-selectable):**
- Default below: `cagliostrolab/animagine-xl-3.1` (Animagine XL 3.1 — strong anime/manga line styles; SDXL).
- Alternatives you may set in `MODEL_ID`: `Linaqruf/animagine-xl-2.0`, `gsdf/Counterfeit-V2.5`, or another open weight you accept for licensing.

Large weights download to the Colab disk — do not commit them to git.


In [ ]:
# Lightweight helpers
%pip install -q pillow pydantic python-dotenv

# GPU stack (required for REAL generation — do not skip)
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q diffusers transformers accelerate safetensors

import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

print("Dependencies installed.")


## 2. Check GPU

If CUDA is unavailable, this cell prints **`COLAB_GPU_BLOCKED`** and raises so later cells do not run silent fakes.


In [ ]:
try:
    import torch
except Exception as exc:
    print("COLAB_GPU_BLOCKED")
    raise SystemExit(
        "COLAB_GPU_BLOCKED: torch is not importable. "
        "Switch to a GPU runtime and re-run install. "
        f"({type(exc).__name__}: {exc})"
    ) from exc

GPU = bool(torch.cuda.is_available())
print(f"torch={torch.__version__}")
print(f"CUDA available: {GPU}")
if GPU:
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("COLAB_GPU_BLOCKED")
    raise SystemExit(
        "COLAB_GPU_BLOCKED: No CUDA GPU. "
        "Runtime → Change runtime type → GPU, then re-run. "
        "This notebook never generates mock/Pillow placeholders."
    )


## 3. Load Kaito character bible JSON

Either:
1. **Upload** `character-bible.json` from `characters/kaito/character-bible.json` to `/content/character-bible.json`, or
2. **Paste** JSON into `PASTE_BIBLE_JSON` below (leave empty string to use upload).

Prefer the real project bible for production-quality consistency.


In [ ]:
WORKDIR = Path("/content/echo_inkwell_kaito")
OUT = WORKDIR / "outputs"
WORKDIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Paste full character-bible.json here (or leave "" and upload the file).
PASTE_BIBLE_JSON = ""

bible_path = Path("/content/character-bible.json")
bible = None

if PASTE_BIBLE_JSON.strip():
    bible = json.loads(PASTE_BIBLE_JSON)
    bible_path.write_text(json.dumps(bible, indent=2), encoding="utf-8")
    print("Loaded bible from PASTE_BIBLE_JSON")
elif bible_path.is_file():
    bible = json.loads(bible_path.read_text(encoding="utf-8"))
    print(f"Loaded bible from {bible_path}")
else:
    try:
        from google.colab import files

        print("Upload characters/kaito/character-bible.json …")
        uploaded = files.upload()
        for name, data in uploaded.items():
            if name.endswith(".json"):
                bible_path.write_bytes(data)
                bible = json.loads(data.decode("utf-8"))
                print(f"Loaded uploaded {name}")
                break
    except Exception as exc:
        print(f"Upload UI unavailable ({type(exc).__name__}: {exc})")

if not isinstance(bible, dict):
    raise SystemExit(
        "No Kaito character bible loaded. Paste JSON into PASTE_BIBLE_JSON "
        "or upload character-bible.json to /content/character-bible.json."
    )

print(f"Character: {bible.get('name', '?')} age={bible.get('age', '?')}")
print(f"Keys: {sorted(bible.keys())[:12]}…")


## 4. Build coloring-book line-art prompts + generate A–D

Uses an open anime/manga-capable Diffusers pipeline on GPU only.
Each candidate writes `candidate_X.png` + sidecar metadata with `source_type=REAL` and `production_eligible=true`.

**No mock path.** If the model fails to load or generate, the cell fails loudly.


In [ ]:
from diffusers import AutoPipelineForText2Image

# --- User-selectable open model (anime/manga capable) ---
MODEL_ID = "cagliostrolab/animagine-xl-3.1"
# Other open options (verify license yourself):
# MODEL_ID = "Linaqruf/animagine-xl-2.0"
# MODEL_ID = "gsdf/Counterfeit-V2.5"

WIDTH = 768
HEIGHT = 1024
SEED_BASE = 12042026  # change for a fresh exploration batch
NUM_STEPS = 28
GUIDANCE = 7.0

MASTER_VISUAL_STYLE = (
    "Clean black-and-white manga line art for a print coloring-book style children's manga. "
    "Bold, closed outlines; flat white background areas ready for coloring; clear silhouettes; "
    "consistent character proportions; expressive but simple faces; readable panel-friendly compositions; "
    "no screentones; no gray washes; no painterly shading; high contrast ink lines only."
)

COLORING_BOOK_RULES = """COLORING BOOK RULES:
- Pure white backgrounds / open regions suitable for hand coloring
- Closed shapes with continuous black outlines
- No filled black areas larger than hair/eyes accents unless specified
- No grayscale gradients, hatching-as-shade, or photographic texture
- Leave clear negative space for speech balloons and captions (text added later)
- Keep important details inside safe margins"""

NEGATIVE = (
    "photorealistic, 3d render, cgi, blurry, lowres, watermark, signature, logo, "
    "text, letters, words, speech bubble text, caption text, speech balloons with writing, "
    "color, colored, watercolor, airbrush, soft shading, gradient, grayscale wash, "
    "screentone, moire, noise, jpeg artifacts, extra limbs, deformed hands, "
    "nsfw, gore, horror, creepy uncanny faces"
)

DESIGN_VARIATIONS = {
    "A": (
        "Kaito master design exploration A: confident curious; full-body + face; hood down. "
        "14-year-old boy, messy spiky hair, hooded jacket, trousers, sneakers, "
        "manga B&W line art coloring-book style, white background."
    ),
    "B": (
        "Kaito master design exploration B: thoughtful soft expression; bust + full-body. "
        "14-year-old boy, messy spiky hair, hooded jacket, trousers, sneakers, "
        "manga B&W line art coloring-book style, white background."
    ),
    "C": (
        "Kaito master design exploration C: adventurous grin; hooded jacket; cargo trousers. "
        "14-year-old boy, messy spiky hair, hooded jacket, trousers, sneakers, "
        "manga B&W line art coloring-book style, white background."
    ),
    "D": (
        "Kaito master design exploration D: focused creative; holding sketchbook. "
        "14-year-old boy, messy spiky hair, hooded jacket, trousers, sneakers, "
        "manga B&W line art coloring-book style, white background."
    ),
}


def build_prompt(bible: dict, label: str) -> str:
    appearance = bible.get("prompt") or bible.get("appearance") or ""
    parts = [
        MASTER_VISUAL_STYLE,
        COLORING_BOOK_RULES,
        (
            "Professional Japanese manga protagonist CHARACTER DESIGN SHEET. "
            "Black and white clean ink line art only. Pure white background. "
            "No grayscale, no gradients, no color, no screentones, no painted shading. "
            "Coloring-book compatible. Single character: Kaito."
        ),
        DESIGN_VARIATIONS[label],
        (
            "Include readable full-body figure plus a larger face/head detail in the same sheet "
            "composition without written labels or text of any kind."
        ),
        f"Name: {bible.get('name', 'Kaito')}",
        f"Age: {bible.get('age', 14)} — unmistakably a 14-year-old boy, NOT an adult.",
        str(appearance),
    ]
    for key in (
        "face_shape",
        "eye_shape",
        "hairstyle",
        "hair_silhouette",
        "approximate_height",
        "body_proportions",
    ):
        if bible.get(key):
            parts.append(f"{key}: {bible[key]}")
    clothing = bible.get("clothing") or {}
    if isinstance(clothing, dict) and clothing.get("default_outfit"):
        parts.append(f"outfit: {clothing['default_outfit']}")
    jacket = bible.get("jacket_design") or {}
    if isinstance(jacket, dict):
        parts.append(f"jacket: {json.dumps(jacket)}")
    personality = bible.get("personality")
    if personality:
        parts.append(f"personality cues in posing: {personality}")
    constants = bible.get("must_remain_constant") or []
    if constants:
        parts.append("Must remain constant: " + "; ".join(str(c) for c in constants))
    parts.append(
        "Distinctive recognizable silhouette. Relatable creative teen, not a superhero. "
        "No speech bubbles, no captions, no watermarks, no logos, no gibberish text."
    )
    return "\n\n".join(p for p in parts if p)


print(f"Loading REAL open model: {MODEL_ID} …")
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)
pipe.to("cuda")
print("Model ready on CUDA.")

candidates = []
labels = ("A", "B", "C", "D")

for i, label in enumerate(labels):
    positive = build_prompt(bible, label)
    seed = SEED_BASE + i * 97
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    image = pipe(
        prompt=positive,
        negative_prompt=NEGATIVE,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]

    png_name = f"candidate_{label}.png"
    out_path = OUT / png_name
    image.save(out_path, format="PNG")

    meta = {
        "label": label,
        "character_id": "kaito",
        "kind": "kaito_master_design",
        "backend": "colab",
        "model": MODEL_ID,
        "seed": seed,
        "width": WIDTH,
        "height": HEIGHT,
        "num_inference_steps": NUM_STEPS,
        "guidance_scale": GUIDANCE,
        "source_type": "REAL",
        "production_eligible": True,
        "positive_prompt": positive,
        "negative_prompt": NEGATIVE,
        "output": png_name,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "license_notes": (
            f"Owner must review commercial terms for {MODEL_ID} before KDP production."
        ),
    }
    (OUT / f"candidate_{label}.meta.json").write_text(
        json.dumps(meta, indent=2), encoding="utf-8"
    )
    candidates.append(meta)
    print(f"OK [{label}] seed={seed} -> {out_path.name} source_type=REAL")

manifest = {
    "project": "Echo of the Inkwell",
    "phase": 12,
    "kind": "kaito_master_design",
    "backend": "colab",
    "model": MODEL_ID,
    "source_type": "REAL",
    "production_eligible": True,
    "license_notes": (
        f"Owner must review commercial terms for {MODEL_ID} before KDP production."
    ),
    "candidates": candidates,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(WORKDIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Wrote {len(candidates)} REAL candidates + manifest.")


## 5. Package zip for download + local import

Download the zip, then on your machine:

```bash
python scripts/import_colab_kaito_designs.py path/to/kaito_designs_*.zip
# after visual review:
python scripts/import_colab_kaito_designs.py path/to/kaito_designs_*.zip --mark-production-eligible
```

Review in Streamlit (`streamlit run app.py` → **Kaito Master Design**). Do **not** auto-select. Update `reports/model-licensing.json` with the model id you actually used.


In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
zip_path = WORKDIR / f"kaito_designs_colab_{stamp}.zip"

readme = WORKDIR / "EXPORT_README.txt"
readme.write_text(
    "Echo of the Inkwell — Phase 12 Kaito master design Colab export\n"
    f"Created: {stamp}\n"
    "source_type=REAL production_eligible=true (candidates only — not auto-selected)\n"
    "\n"
    "Import locally:\n"
    "  python scripts/import_colab_kaito_designs.py path/to/this.zip\n"
    "  python scripts/import_colab_kaito_designs.py path/to/this.zip --mark-production-eligible\n"
    "\n"
    "Then review in Streamlit → Kaito Master Design. Never auto-select.\n",
    encoding="utf-8",
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(WORKDIR)))
    zf.write(WORKDIR / "manifest.json", arcname="manifest.json")
    zf.write(readme, arcname="EXPORT_README.txt")

print(f"Package ready: {zip_path}")
print("Import: python scripts/import_colab_kaito_designs.py", zip_path.name)

try:
    from google.colab import files

    files.download(str(zip_path))
except Exception:
    print("Not inside Colab UI — zip left on disk for manual download.")
